In [ ]:
suppressPackageStartupMessages({
    library(monocle3)
    library(MASS)
    library(reshape2)
    library(dplyr)
    library(ggplot2)
    library(grid)
    library(gridExtra)
    library(Matrix)
    library(IRdisplay)
    library(tibble)
    library(ggridges)
    library(pheatmap)
    library(stringr)
    library(scales)
    library(DelayedArray)
    library(hooke)
    library(splines)
    library(msigdbr)
    library(rmcorr)
    library(ComplexHeatmap)
    library(circlize)
    library(viridis)
    library(glmmTMB)
    library(ggeffects)
})

In [ ]:
aggregate_gene_score = function(cds, gene_list=c("CCNB1", "CCNB2","CDK1"), name = "gene_list_score"){
  cds_gene_list_norm = cds[rowData(cds)$gene_short_name %in% gene_list,] 
  aggregate_marker_expression = counts(cds_gene_list_norm)
  aggregate_marker_expression = t(t(aggregate_marker_expression) / colData(cds_gene_list_norm)$Size_Factor)
  aggregate_marker_expression = Matrix::colSums(aggregate_marker_expression)
  aggregate_marker_expression = log(aggregate_marker_expression + 1)
  colData(cds)[[name]] = aggregate_marker_expression
  return(cds)
}

estimate_cell_cycle = function(cds, g1s_markers, g2m_markers){
  cds_g1s = cds[rowData(cds)$gene_short_name %in% g1s_markers,] 
  aggregate_g1s_expression = counts(cds_g1s)
  aggregate_g1s_expression = t(t(aggregate_g1s_expression) / colData(cds_g1s)$Size_Factor)
  aggregate_g1s_expression = Matrix::colSums(aggregate_g1s_expression)
  
  cds_g2m = cds[rowData(cds)$gene_short_name %in% g2m_markers,] 
  aggregate_g2m_expression = counts(cds_g2m)
  aggregate_g2m_expression = t(t(aggregate_g2m_expression) / colData(cds_g2m)$Size_Factor)
  aggregate_g2m_expression = Matrix::colSums(aggregate_g2m_expression)
  #pData(cds)$cell_cycle_score = aggregate_g1s_expression
  cds$g1s_score = log(aggregate_g1s_expression+1)
  cds$g2m_score = log(aggregate_g2m_expression+1)
  cds$proliferation_index = log(aggregate_g1s_expression + aggregate_g2m_expression + 1)
  return(cds)
}

pseudobulk_cds = function(cds_sub, sample_group = "Cluster_group", cell_group = "Grouping") {
    cf_ccs = new_cell_count_set(cds_sub,
                            sample_group = sample_group,
                            cell_group = cell_group)
    
    pb_cds = hooke:::pseudobulk_ccs_for_states(cf_ccs)
    
    return(pb_cds)
    
}

normalize <- function(x) {
    return((x- min(x)) /(max(x)-min(x)))
}

# Combine data

In [ ]:
zfish_dir = '/net/seahub_zfish/vol1/data/reference_cds/v2.2.0/reference_cds'
cds_fish = load_monocle_objects(directory_path = zfish_dir)

In [ ]:
cds_fish

In [ ]:
colnames(colData(cds_fish))

In [ ]:
umap_coordinates = as.data.frame(reducedDims(cds_fish)[['UMAP']])
colnames(umap_coordinates) = c("UMAP_1", "UMAP_2", "UMAP_3")
head(umap_coordinates)

In [ ]:
dim(cds_fish)

In [ ]:
cds_fish$UMAP_1 = umap_coordinates$UMAP_1
cds_fish$UMAP_2 = umap_coordinates$UMAP_2
cds_fish$UMAP_3 = umap_coordinates$UMAP_3

# Assign scores

In [ ]:
rowdata = as.data.frame(rowData(cds_fish))

In [ ]:
cc.genes = readRDS("../data/cc.genes.RDS")
model_glmm_0.7_0.4 = readRDS("../data/nucleolar_model_glmm_Z5_0.7_0.4_n_6.RDS")
res_df_0.7_0.4 = readRDS("../data/nucleolar_genes_Z5_0.7_0.4_n_6.RDS")

In [ ]:
res_df_0.7_0.4 = res_df_0.7_0.4 %>%
    mutate(gene_short_name = case_match(as.character(gene_short_name),
        "AI314180" ~ "Ecpas",
        .default = gene_short_name
))

In [ ]:
gene_df_0.7_0.4 <- orthogene::convert_orthologs(gene_df = as.character(res_df_0.7_0.4$gene_short_name), 
                                        method = "homologene",
                                        as_sparse = F,
                                        gene_input = "gene_ID", 
                                        gene_output = "columns", 
                                        input_species = "mouse",
                                        output_species = "zebrafish",
                                        non121_strategy = "keep_both_species") 

In [ ]:
non_mapped_genes = res_df_0.7_0.4 %>%
    mutate(input_gene = as.character(gene_short_name)) %>%
    dplyr::select(input_gene, Spearman) %>%
    left_join(., gene_df_0.7_0.4) %>% filter(is.na(ortholog_gene))

non_mapped_genes

In [ ]:
mapped_but_not_in_rowdata = gene_df_0.7_0.4 %>%
    filter(!ortholog_gene %in% (rowdata %>%
                            filter(gene_short_name %in% gene_df_0.7_0.4$ortholog_gene) %>%
                            pull(gene_short_name)))
mapped_genes = rowdata %>%
                    filter(gene_short_name %in% gene_df_0.7_0.4$ortholog_gene) %>%
                    pull(gene_short_name)

mapped_but_not_in_rowdata

In [ ]:
remapped_genes = rowdata %>%
    filter((gene_short_name %in% tolower(non_mapped_genes$input_gene)) |
           (grepl("rbm25", gene_short_name)) | 
           (grepl("larp4a|stt3a|ubap2a", gene_short_name)))


remapped_genes

In [ ]:
nucleolar_genes_manually_mapped = c(mapped_genes, remapped_genes$gene_short_name)
length(nucleolar_genes_manually_mapped)
sort(nucleolar_genes_manually_mapped)

In [ ]:
cc.genes = readRDS("../data/cc.genes.RDS")

In [ ]:
zfish_s_genes <- orthogene::convert_orthologs(gene_df = cc.genes$s.genes,
                                              method = "homologene",
                                        gene_input = "gene_ID", 
                                        gene_output = "columns", 
                                        input_species = "human",
                                        output_species = "zebrafish",
                                        non121_strategy = "drop_output_species") 

zfish_g2m_genes <- orthogene::convert_orthologs(gene_df = cc.genes$g2m.genes,
                                                method = "homologene",
                                        gene_input = "gene_ID", 
                                        gene_output = "columns", 
                                        input_species = "human",
                                        output_species = "zebrafish",
                                        non121_strategy = "drop_output_species") 

In [ ]:
mapped_but_not_in_rowdata = zfish_s_genes %>%
    filter(!ortholog_gene %in% (rowdata %>%
                            filter(gene_short_name %in% zfish_s_genes$ortholog_gene) %>%
                            pull(gene_short_name)))
mapped_genes = rowdata %>%
                    filter(gene_short_name %in% zfish_s_genes$ortholog_gene) %>%
                    pull(gene_short_name)

mapped_but_not_in_rowdata

In [ ]:
non_mapped_genes = cc.genes$s.genes[!cc.genes$s.genes %in% zfish_s_genes$input_gene ]
non_mapped_genes

In [ ]:
remapped_genes_s = rowdata %>%
    filter((gene_short_name %in% tolower(non_mapped_genes)) |
           (grepl("atad2$", gene_short_name)) | 
           grepl("ENSDARG00000088924", id))

## ENSDARG00000088924 = "MLF1IP"

s_genes = c(mapped_genes, remapped_genes_s$gene_short_name)

length(s_genes)
s_genes

In [ ]:
mapped_but_not_in_rowdata = zfish_g2m_genes %>%
    filter(!ortholog_gene %in% (rowdata %>%
                            filter(gene_short_name %in% zfish_g2m_genes$ortholog_gene) %>%
                            pull(gene_short_name)))
mapped_genes = rowdata %>%
                    filter(gene_short_name %in% zfish_g2m_genes$ortholog_gene) %>%
                    pull(gene_short_name)

mapped_but_not_in_rowdata

In [ ]:
non_mapped_genes = cc.genes$g2m.genes[!cc.genes$g2m.genes %in% zfish_g2m_genes$input_gene ]
non_mapped_genes

In [ ]:
remapped_genes_g2m = rowdata %>%
    filter((gene_short_name %in% tolower(non_mapped_genes)) |
           (grepl("jpt1a", gene_short_name)) | 
           grepl("ENSDARG00000043004|ENSDARG00000070656|ENSDARG00000068759", id))

## jpt1a = hn1a
## ENSDARG00000043004 = "HJURP"
## ENSDARG00000070656 = "CDCA3"
## ENSDARG00000068759 = "CDCA2"


g2m_genes = c(mapped_genes, remapped_genes_g2m$gene_short_name)

length(g2m_genes)
g2m_genes

In [ ]:
cds_fish = estimate_cell_cycle(cds_fish, 
                               g1s_markers = s_genes, 
                               g2m_markers = g2m_genes)

In [ ]:
cds_fish = aggregate_gene_score(cds_fish, nucleolar_genes_manually_mapped, 
                                            "nucleolar_genes_manually_mapped")

In [ ]:
s_name <- "0.7_0.4"

model_file <- paste0("nucleolar_model_glmm_Z5_", s_name, "_n_6.RDS")
temp_model <- readRDS(model_file)


pred_col       <- paste0("score_pred_glmm_", s_name, "_remapped")
raw_score_col  <- "nucleolar_genes_manually_mapped"
norm_score_col <- "normalized_nuc_score_remapped"

temp_model <- readRDS(model_file)

cds_fish[[norm_score_col]] <- normalize(cds_fish[[raw_score_col]])

target_var <- all.vars(formula(temp_model))[2] 
pred_df <- data.frame(cds_fish[[norm_score_col]])
colnames(pred_df) <- target_var


cds_fish[[pred_col]] <- as.double(predict(temp_model, 
                                        newdata = pred_df, 
                                        re.form = NA, 
                                        type = "response"))


In [ ]:
gene_set_BP = msigdbr::msigdbr(species = "Danio rerio", subcategory = "GO:BP")
gene_set_hallmarks = msigdbr::msigdbr(species = "Danio rerio", category = "H")
gene_set_reactome = msigdbr::msigdbr(species = "Danio rerio", subcategory = "CP:REACTOME")
gene_set_cgp = msigdbr::msigdbr(species = "Danio rerio", subcategory = "CGP")
gene_set_kegg = msigdbr::msigdbr(species = "Danio rerio", subcategory = "CP:KEGG_MEDICUS")

In [ ]:
gene_set_BP_df = gene_set_BP %>% 
                dplyr::distinct(gs_name, gene_short_name=gene_symbol) %>% 
                as.data.frame()

gene_set_hallmarks_df = gene_set_hallmarks %>% 
                dplyr::distinct(gs_name, gene_short_name=gene_symbol) %>% 
                as.data.frame()

gene_set_kegg_df = gene_set_kegg %>% 
                dplyr::distinct(gs_name, gene_short_name=gene_symbol) %>% 
                as.data.frame()

In [ ]:
granule_genes = gene_set_BP_df %>%
    filter(grepl("GOBP_STRESS_GRANULE_ASSEMBLY", gs_name)) %>%
    distinct(gene_short_name, .keep_all=T) %>%
    mutate(gene_short_name = ifelse(gene_short_name == "g3bp2a", "g3bp2", gene_short_name))

In [ ]:
upr_genes = gene_set_hallmarks_df %>%
    filter(grepl("UNFOLDED", gs_name)) %>%
    distinct(gene_short_name, .keep_all=T)

In [ ]:
proteasome_genes = gene_set_kegg_df %>%
    filter(grepl("KEGG_MEDICUS_REFERENCE_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION", gs_name)) %>%
    distinct(gene_short_name, .keep_all=T)

In [ ]:
cds_fish = aggregate_gene_score(cds_fish, granule_genes$gene_short_name, "granule_score")
cds_fish = aggregate_gene_score(cds_fish, upr_genes$gene_short_name, "upr_score")
cds_fish = aggregate_gene_score(cds_fish, proteasome_genes$gene_short_name, "ups_score")

In [ ]:
save_monocle_objects(cds_fish, directory_path = "../data/cds_ref_v2.2.0_glmm")